# Notebook 04 — Feature Engineering

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

Konstruujemy kompletny zestaw cech (feature set) dla modeli ML. Każda cecha ma ekonomiczne uzasadnienie:

- **Cechy BTC (btc_*):** płynność, zmienność, momentum, wolumen — sygnały stresu na rynku kryptowalut
- **Cechy WIG20 (wig_*):** analogiczne dla GPW — będziemy prognozować stres płynności GPW
- **Cross-market (cross_*):** korelacje i beta między rynkami — kanały transmisji
- **Makro (macro_*):** zmienne kontrolne — globalne nastroje (VIX), siła dolara (DXY)

## Zmienna docelowa
- `target_liquidity_stress` = 1 jeśli w ciągu 5 dni: spadek WIG20 > 2% LUB Amihud > kwantyl 90%
- `target_severe_stress` = 1 jeśli spadek WIG20 > 4% (analiza wrażliwości)

In [1]:
# ============================================================
# Reprodukowalność + Importy
# ============================================================
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import warnings
from pathlib import Path
from scipy import stats
from scipy.stats import pointbiserialr
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.6f}'.format)

# ── Ścieżki ─────────────────────────────────────────────────
BASE_DIR = Path('.')
DATA_PROC = BASE_DIR / 'data' / 'processed'
RESULTS_TABLES = BASE_DIR / 'results' / 'tables'
RESULTS_FIGS = BASE_DIR / 'results' / 'figures'

for d in [DATA_PROC, RESULTS_TABLES, RESULTS_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Styl ─────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.size'] = 11
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['savefig.dpi'] = 300

COLOR_BTC = '#F7931A'
COLOR_WIG = '#003087'
COLOR_HRI = '#C0392B'

print("Importy zakonczone.")

Importy zakonczone.


In [2]:
# ── Wczytanie master dataset ─────────────────────────────────
master_path = DATA_PROC / 'master_dataset.csv'
df = pd.read_csv(master_path, index_col='Date', parse_dates=True)

print(f"Master dataset: {df.shape}")
print(f"Zakres: {df.index.min().date()} → {df.index.max().date()}")
print(f"Kolumny: {list(df.columns)}")

Master dataset: (1906, 24)
Zakres: 2019-01-02 → 2026-08-14
Kolumny: ['wig_open', 'wig_high', 'wig_low', 'wig_close', 'wig_volume', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'vix', 'dxy', 'fed_rate', 'flag_btc_outlier', 'flag_wig_outlier', 'btc_log_return', 'wig_log_return', 'vix_change', 'dxy_change', 'btc_1h_return_mean', 'btc_1h_return_std', 'btc_1h_return_min', 'btc_1h_return_max', 'btc_1h_n_obs']


## Funkcje pomocnicze do obliczeń technicznych

In [3]:
# ============================================================
# Funkcje pomocnicze — wskaźniki techniczne
# ============================================================

def compute_rsi(series, period=14):
    """
    Relative Strength Index (RSI).
    Wartości: 0-100. >70 = wykupienie, <30 = wyprzedanie.
    """
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period, min_periods=period).mean()
    avg_loss = loss.rolling(period, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(series, fast=12, slow=26, signal=9):
    """
    MACD (Moving Average Convergence Divergence).
    Returns: macd_line, signal_line, histogram
    """
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram


def compute_bollinger_bands(series, window=20, n_std=2):
    """
    Wstęgi Bollingera — szerokość = (upper - lower) / middle.
    Wysoka szerokość = wysoka zmienność.
    """
    ma = series.rolling(window).mean()
    std = series.rolling(window).std()
    upper = ma + n_std * std
    lower = ma - n_std * std
    bb_width = (upper - lower) / ma
    return ma, upper, lower, bb_width


def compute_amihud(returns, volume):
    """
    Wskaźnik Amihuda — miara nielikwidności.
    ILLIQ = |r_t| / Volume_t (× skalujące stałe)
    Wyższe wartości = niższa płynność (wyższy koszt handlu)
    """
    # Podstawowa formuła: |zwrot| / wolumen
    illiq = np.abs(returns) / volume.replace(0, np.nan)
    # Skalowanie: *1e6 żeby wartości były czytelne
    return illiq * 1e6


def compute_parkinson_vol(high, low):
    """
    Estymator zmienności Parkinsona.
    Bardziej wydajny niż close-to-close dla cen intraday.
    Formuła: 1/(4*ln(2)) * (ln(H/L))^2
    """
    log_hl = np.log(high / low)
    parkinson = (log_hl ** 2) / (4 * np.log(2))
    return parkinson


def compute_rolling_beta(returns_x, returns_y, window=60):
    """
    Rolling beta: cov(X,Y) / var(Y)
    Interpretacja: beta BTC względem WIG20 — wrażliwość BTC na zmiany WIG20
    """
    cov = returns_x.rolling(window).cov(returns_y)
    var = returns_y.rolling(window).var()
    return cov / var.replace(0, np.nan)


print("Funkcje pomocnicze zdefiniowane:")
print("  - compute_rsi()")
print("  - compute_macd()")
print("  - compute_bollinger_bands()")
print("  - compute_amihud()")
print("  - compute_parkinson_vol()")
print("  - compute_rolling_beta()")

Funkcje pomocnicze zdefiniowane:
  - compute_rsi()
  - compute_macd()
  - compute_bollinger_bands()
  - compute_amihud()
  - compute_parkinson_vol()
  - compute_rolling_beta()


## 1. Cechy BTC

Budujemy 20+ cech opisujących dynamikę, płynność i zmienność Bitcoina.

In [4]:
# ============================================================
# KROK 1: Cechy BTC
# ============================================================

feat = pd.DataFrame(index=df.index)

print("Obliczanie cech BTC...")

# ─── 1.1 Log-zwroty ──────────────────────────────────────────
feat['btc_log_return'] = np.log(df['btc_close'] / df['btc_close'].shift(1))
print(f"  btc_log_return: {feat['btc_log_return'].notna().sum()} wartości")

# ─── 1.2 Statystyki zwrotów 1H (jeśli dostępne) ──────────────
for col_1h in ['btc_1h_return_mean', 'btc_1h_return_std',
               'btc_1h_return_min', 'btc_1h_return_max']:
    if col_1h in df.columns:
        feat[col_1h] = df[col_1h]
    else:
        feat[col_1h] = np.nan
print(f"  Statystyki 1H: {feat['btc_1h_return_mean'].notna().sum()} wartości")

# ─── 1.3 Zmienność rolling ────────────────────────────────────
for window in [7, 14, 21, 30]:
    feat[f'btc_realized_vol_{window}d'] = (
        feat['btc_log_return'].rolling(window).std() * np.sqrt(252)
    )
print(f"  Realized vol (7/14/21/30d): OK")

# ─── 1.4 Amihud i Parkinson ───────────────────────────────────
if 'btc_volume' in df.columns:
    btc_vol = df['btc_volume'].replace(0, np.nan)
    feat['btc_amihud'] = compute_amihud(feat['btc_log_return'], btc_vol)
    feat['btc_amihud_ma30'] = feat['btc_amihud'].rolling(30).mean()
    feat['btc_volume_normalized'] = df['btc_volume'] / df['btc_volume'].rolling(30).mean()
    print(f"  Amihud BTC: OK ({feat['btc_amihud'].notna().sum()} wartości)")
else:
    feat['btc_amihud'] = np.nan
    feat['btc_amihud_ma30'] = np.nan
    feat['btc_volume_normalized'] = np.nan
    print(f"  Amihud BTC: brak danych wolumenu")

if all(c in df.columns for c in ['btc_high', 'btc_low']):
    feat['btc_parkinson_vol'] = compute_parkinson_vol(df['btc_high'], df['btc_low'])
    print(f"  Parkinson vol BTC: OK")
else:
    feat['btc_parkinson_vol'] = np.nan
    print(f"  Parkinson vol BTC: brak danych High/Low")

# ─── 1.5 RSI ─────────────────────────────────────────────────
feat['btc_rsi_14'] = compute_rsi(df['btc_close'], period=14)
print(f"  RSI(14) BTC: OK")

# ─── 1.6 MACD ────────────────────────────────────────────────
macd, macd_sig, macd_hist = compute_macd(df['btc_close'])
feat['btc_macd'] = macd
feat['btc_macd_signal'] = macd_sig
feat['btc_macd_hist'] = macd_hist
print(f"  MACD(12,26,9) BTC: OK")

# ─── 1.7 Bollinger Bands Width ────────────────────────────────
_, _, _, bb_width = compute_bollinger_bands(df['btc_close'], window=20, n_std=2)
feat['btc_bb_width'] = bb_width
print(f"  Bollinger Band Width (20d): OK")

# ─── 1.8 Momenty wyższych rzędów ─────────────────────────────
feat['btc_rolling_skew_30d'] = feat['btc_log_return'].rolling(30).skew()
feat['btc_rolling_kurt_30d'] = feat['btc_log_return'].rolling(30).kurt()
print(f"  Rolling skewness/kurtosis (30d): OK")

# ─── 1.9 MA(200) dummy ────────────────────────────────────────
ma200 = df['btc_close'].rolling(200).mean()
feat['btc_price_above_ma200'] = (df['btc_close'] > ma200).astype(int)
feat['btc_dist_from_ma200'] = (df['btc_close'] - ma200) / ma200  # relatywna odległość
print(f"  MA(200) dummy: OK")

# ─── 1.10 Lag features ────────────────────────────────────────
for lag in [1, 2, 3, 5, 10]:
    feat[f'btc_log_return_lag{lag}'] = feat['btc_log_return'].shift(lag)
print(f"  Lag features (1,2,3,5,10): OK")

print(f"\nCechy BTC: {[c for c in feat.columns if c.startswith('btc_')]}")
print(f"Łącznie cech BTC: {sum(1 for c in feat.columns if c.startswith('btc_'))}")

Obliczanie cech BTC...
  btc_log_return: 1905 wartości
  Statystyki 1H: 0 wartości
  Realized vol (7/14/21/30d): OK
  Amihud BTC: OK (1905 wartości)
  Parkinson vol BTC: OK
  RSI(14) BTC: OK
  MACD(12,26,9) BTC: OK
  Bollinger Band Width (20d): OK
  Rolling skewness/kurtosis (30d): OK
  MA(200) dummy: OK
  Lag features (1,2,3,5,10): OK

Cechy BTC: ['btc_log_return', 'btc_1h_return_mean', 'btc_1h_return_std', 'btc_1h_return_min', 'btc_1h_return_max', 'btc_realized_vol_7d', 'btc_realized_vol_14d', 'btc_realized_vol_21d', 'btc_realized_vol_30d', 'btc_amihud', 'btc_amihud_ma30', 'btc_volume_normalized', 'btc_parkinson_vol', 'btc_rsi_14', 'btc_macd', 'btc_macd_signal', 'btc_macd_hist', 'btc_bb_width', 'btc_rolling_skew_30d', 'btc_rolling_kurt_30d', 'btc_price_above_ma200', 'btc_dist_from_ma200', 'btc_log_return_lag1', 'btc_log_return_lag2', 'btc_log_return_lag3', 'btc_log_return_lag5', 'btc_log_return_lag10']
Łącznie cech BTC: 27


## 2. Cechy WIG20

In [5]:
# ============================================================
# KROK 2: Cechy WIG20
# ============================================================

print("Obliczanie cech WIG20...")

# Log-zwroty
feat['wig_log_return'] = np.log(df['wig_close'] / df['wig_close'].shift(1))

# Amihud WIG20
if 'wig_volume' in df.columns:
    wig_vol = df['wig_volume'].replace(0, np.nan)
    wig_vol_valid = wig_vol[wig_vol > 100]  # usuń podejrzanie małe wartości
    feat['wig_amihud'] = compute_amihud(feat['wig_log_return'], wig_vol)
    feat['wig_amihud_ma30'] = feat['wig_amihud'].rolling(30).mean()
    feat['wig_volume_normalized'] = df['wig_volume'] / df['wig_volume'].rolling(30).mean()
    print(f"  Amihud WIG20: OK ({feat['wig_amihud'].notna().sum()} wartości)")
else:
    # Brak wolumenu WIG20 — proxy: odwrócony wolumen BTC (heurystyka)
    print("  UWAGA: Brak wolumenu WIG20 — konstruuję proxy Amihuda z High/Low spread")
    if all(c in df.columns for c in ['wig_high', 'wig_low', 'wig_close']):
        # Używamy Parkinsona jako proxy nielikwidności
        feat['wig_amihud'] = compute_parkinson_vol(df['wig_high'], df['wig_low']) * 1000
        feat['wig_amihud_ma30'] = feat['wig_amihud'].rolling(30).mean()
    else:
        # Ostateczny proxy: rolling std zwrotów jako miara niepewności
        feat['wig_amihud'] = feat['wig_log_return'].rolling(5).std() * 1000
        feat['wig_amihud_ma30'] = feat['wig_amihud'].rolling(30).mean()
    feat['wig_volume_normalized'] = np.nan
    print(f"  Amihud WIG20 (proxy): {feat['wig_amihud'].notna().sum()} wartości")

# Parkinson WIG20
if all(c in df.columns for c in ['wig_high', 'wig_low']):
    feat['wig_parkinson_vol'] = compute_parkinson_vol(df['wig_high'], df['wig_low'])
    print(f"  Parkinson vol WIG20: OK")
else:
    feat['wig_parkinson_vol'] = np.nan
    print(f"  Parkinson vol WIG20: brak High/Low")

# Realized vol (21d)
feat['wig_realized_vol_21d'] = feat['wig_log_return'].rolling(21).std() * np.sqrt(252)

# RSI
feat['wig_rsi_14'] = compute_rsi(df['wig_close'], period=14)

# Lag features WIG
for lag in [1, 2, 3, 5]:
    feat[f'wig_log_return_lag{lag}'] = feat['wig_log_return'].shift(lag)

print(f"\nCechy WIG20: {[c for c in feat.columns if c.startswith('wig_')]}")
print(f"Łącznie cech WIG20: {sum(1 for c in feat.columns if c.startswith('wig_'))}")

Obliczanie cech WIG20...
  Amihud WIG20: OK (1905 wartości)
  Parkinson vol WIG20: OK

Cechy WIG20: ['wig_log_return', 'wig_amihud', 'wig_amihud_ma30', 'wig_volume_normalized', 'wig_parkinson_vol', 'wig_realized_vol_21d', 'wig_rsi_14', 'wig_log_return_lag1', 'wig_log_return_lag2', 'wig_log_return_lag3', 'wig_log_return_lag5']
Łącznie cech WIG20: 11


## 3. Cechy cross-market

In [6]:
# ============================================================
# KROK 3: Cechy cross-market
# ============================================================

print("Obliczanie cech cross-market...")

btc_ret = feat['btc_log_return']
wig_ret = feat['wig_log_return']

# Rolling correlation
feat['cross_corr_30d'] = btc_ret.rolling(30).corr(wig_ret)
feat['cross_corr_60d'] = btc_ret.rolling(60).corr(wig_ret)
feat['cross_corr_90d'] = btc_ret.rolling(90).corr(wig_ret)
print(f"  Rolling correlation (30/60/90d): OK")

# Rolling beta (BTC vs WIG20)
feat['cross_beta_60d'] = compute_rolling_beta(btc_ret, wig_ret, window=60)
print(f"  Rolling beta (60d): OK")

# Korelacja zmienności
btc_vol_21 = btc_ret.rolling(21).std()
wig_vol_21 = wig_ret.rolling(21).std()
feat['cross_corr_vol_30d'] = btc_vol_21.rolling(30).corr(wig_vol_21)
print(f"  Korelacja zmienności (30d): OK")

# Relative strength BTC vs WIG20
feat['cross_relative_strength'] = (
    (1 + btc_ret).cumprod() / (1 + wig_ret).cumprod()
)
# Normalizuj
feat['cross_relative_strength'] = (
    feat['cross_relative_strength'] / feat['cross_relative_strength'].rolling(60).mean()
)

# Spread zmienności (BTC - WIG) — chwilowe odchylenie od normy
feat['cross_vol_spread_21d'] = btc_vol_21 - wig_vol_21

print(f"\nCechy cross-market: {[c for c in feat.columns if c.startswith('cross_')]}")

Obliczanie cech cross-market...
  Rolling correlation (30/60/90d): OK
  Rolling beta (60d): OK
  Korelacja zmienności (30d): OK

Cechy cross-market: ['cross_corr_30d', 'cross_corr_60d', 'cross_corr_90d', 'cross_beta_60d', 'cross_corr_vol_30d', 'cross_relative_strength', 'cross_vol_spread_21d']


## 4. Cechy makroekonomiczne

In [7]:
# ============================================================
# KROK 4: Cechy makroekonomiczne
# ============================================================

print("Dodawanie cech makroekonomicznych...")

if 'vix' in df.columns:
    feat['macro_vix'] = df['vix']
    feat['macro_vix_change'] = df['vix'].pct_change()
    feat['macro_vix_ma30'] = df['vix'].rolling(30).mean()
    feat['macro_vix_above_30'] = (df['vix'] > 30).astype(int)  # elevated fear
    feat['macro_vix_above_40'] = (df['vix'] > 40).astype(int)  # extreme fear
    print(f"  VIX: OK ({feat['macro_vix'].notna().sum()} wartości)")
else:
    feat['macro_vix'] = np.nan
    feat['macro_vix_change'] = np.nan
    print(f"  VIX: BRAK danych")

if 'dxy' in df.columns:
    feat['macro_dxy'] = df['dxy']
    feat['macro_dxy_change'] = df['dxy'].pct_change()
    feat['macro_dxy_log_return'] = np.log(df['dxy'] / df['dxy'].shift(1))
    print(f"  DXY: OK ({feat['macro_dxy'].notna().sum()} wartości)")
else:
    feat['macro_dxy'] = np.nan
    feat['macro_dxy_change'] = np.nan
    feat['macro_dxy_log_return'] = np.nan
    print(f"  DXY: BRAK danych")

if 'fed_rate' in df.columns:
    feat['macro_fed_rate'] = df['fed_rate']
    feat['macro_fed_rate_change'] = df['fed_rate'].diff()  # 0 lub ±0.25%
    feat['macro_fed_rate_above_4'] = (df['fed_rate'] > 4).astype(int)
    print(f"  Fed Rate: OK ({feat['macro_fed_rate'].notna().sum()} wartości)")
else:
    feat['macro_fed_rate'] = np.nan
    feat['macro_fed_rate_change'] = np.nan
    print(f"  Fed Rate: BRAK danych")

print(f"\nCechy macro: {[c for c in feat.columns if c.startswith('macro_')]}")

Dodawanie cech makroekonomicznych...
  VIX: OK (1906 wartości)
  DXY: OK (1906 wartości)
  Fed Rate: OK (1906 wartości)

Cechy macro: ['macro_vix', 'macro_vix_change', 'macro_vix_ma30', 'macro_vix_above_30', 'macro_vix_above_40', 'macro_dxy', 'macro_dxy_change', 'macro_dxy_log_return', 'macro_fed_rate', 'macro_fed_rate_change', 'macro_fed_rate_above_4']


## 5. Zmienna docelowa (target)

**`target_liquidity_stress` = 1 jeśli w ciągu następnych 5 dni:**
- Skumulowany zwrot WIG20 < -2% **LUB**
- Wskaźnik Amihud WIG20 > kwantyl 90. historyczny

Logika: stres płynności to albo duże spadki cen (wyprzedaż) albo wysoka niepłynność (szeroki spread, trudności z transakcją).

In [8]:
# ============================================================
# KROK 5: Zmienna objaśniana — susza płynnościowa (wersja poprawiona)
# ============================================================
# Wcześniejsza definicja (alternatywa: spadek minimum ceny o 2% LUB przekroczenie
# 90. percentyla Amihuda w oknie 5 sesji, próg liczony na całej próbie) dawała
# klasę pozytywną o liczebności ~50% i zawierała wyciek informacji z przyszłości.
# Obecna definicja opiera się na złożonym wskaźniku warunków płynnościowych
# i progu wyznaczonym wyłącznie na okresie treningowym — szczegóły w METODYKA_ZMIANY.md.

import importlib
import pipeline_lib as pl
importlib.reload(pl)

print("=" * 60)
print("Konstruowanie zmiennej objaśnianej")
print("=" * 60)

HORIZON = pl.HORIZON

liq = pl.build_liquidity_stress_index(df, train_end=pl.TRAIN_END)
for col in ['liq_amihud', 'liq_spread', 'liq_turnover_gap', 'liq_realized_vol',
            'liq_stress_index']:
    feat[col] = liq[col].reindex(feat.index)

print("\nKomponenty wskaznika warunkow plynnosciowych (korelacje):")
zc = [c for c in liq.columns if c.startswith('z_')]
print(liq[zc].corr().round(3).to_string())

target, tau = pl.build_target(liq['liq_stress_index'], horizon=HORIZON,
                              train_end=pl.TRAIN_END, quantile=pl.TARGET_QUANTILE)
feat['target_liquidity_stress'] = target.reindex(feat.index)

print(f"\nProg suszy plynnosciowej (kwantyl {pl.TARGET_QUANTILE} okresu treningowego): {tau:.4f}")
summary = pl.summarize_target(feat['target_liquidity_stress'])
for k, v in summary.items():
    print(f"  {k:14s} {v}")

print("\nRozklad zmiennej objasnianej w latach:")
t = feat['target_liquidity_stress'].dropna()
for year in sorted(set(t.index.year)):
    yr = t[t.index.year == year]
    print(f"  {year}: {int(yr.sum()):3d}/{len(yr):3d} zdarzen ({100*yr.mean():5.1f}%)")

# Analiza wrazliwosci definicji (prog x horyzont) — do podrozdzialu 3.1
sens = []
for q in pl.SENSITIVITY_QUANTILES:
    for h in pl.SENSITIVITY_HORIZONS:
        tt, tt_tau = pl.build_target(liq['liq_stress_index'], horizon=h,
                                     train_end=pl.TRAIN_END, quantile=q)
        sens.append({'Kwantyl progu': q, 'Horyzont (sesje)': h,
                     'Prog': round(tt_tau, 4),
                     'Udzial klasy pozytywnej (%)': round(100 * tt.dropna().mean(), 2),
                     'Liczba zdarzen': int(tt.dropna().sum())})
sens_df = pd.DataFrame(sens)
sens_df.to_csv(RESULTS_TABLES / '04_target_sensitivity.csv', index=False)
print("\nAnaliza wrazliwosci definicji zmiennej objasnianej:")
print(sens_df.to_string(index=False))

n_total = int(feat['target_liquidity_stress'].notna().sum())
n_stress = int(feat['target_liquidity_stress'].sum())
pct_stress = 100 * n_stress / n_total
print(f"\nRAZEM: {n_stress}/{n_total} zdarzen ({pct_stress:.1f}%), "
      f"waga klasy pozytywnej = {(n_total - n_stress) / max(n_stress, 1):.1f}")


Konstruowanie zmiennej objaśnianej

Komponenty wskaznika warunkow plynnosciowych (korelacje):
                    z_liq_amihud  z_liq_spread  z_liq_turnover_gap  z_liq_realized_vol
z_liq_amihud            1.000000      0.119000            0.052000            0.396000
z_liq_spread            0.119000      1.000000            0.020000            0.385000
z_liq_turnover_gap      0.052000      0.020000            1.000000           -0.142000
z_liq_realized_vol      0.396000      0.385000           -0.142000            1.000000

Prog suszy plynnosciowej (kwantyl 0.9 okresu treningowego): 0.5394
  n              1901
  positive       134
  positive_pct   7.05
  train_n        1252
  train_pct      10.06
  val_n          249
  val_pct        0.0
  test_n         400
  test_pct       2.0

Rozklad zmiennej objasnianej w latach:
  2019:   3/248 zdarzen (  1.2%)
  2020:  49/252 zdarzen ( 19.4%)
  2021:   9/251 zdarzen (  3.6%)
  2022:  58/251 zdarzen ( 23.1%)
  2023:   7/250 zdarzen (  2.8%)
  20

In [9]:
# ============================================================
# KROK 6: Analiza brakujących wartości i czyszczenie
# ============================================================

print("Analiza brakujących wartości w zbiorze cech:")
print("=" * 60)

nan_summary = feat.isnull().sum()
nan_pct = (nan_summary / len(feat) * 100).round(2)

nan_report = pd.DataFrame({
    'N brakujących': nan_summary,
    '% brakujących': nan_pct
}).sort_values('% brakujących', ascending=False)

print(nan_report[nan_report['N brakujących'] > 0].to_string())

# Forward-fill dla brakujących (max 5 dni) — WYŁĄCZNIE dla cech.
# Kolumna celu nie może być uzupełniana: ostatnie HORIZON sesji nie mają okna
# przyszłości, więc ich etykieta jest nieznana i musi pozostać NaN. Wcześniej
# ffill obejmował całą ramkę i przeciągał ostatnią znaną etykietę na dokładnie
# te sesje (limit=5 = HORIZON), tworząc pięć wymyślonych zer, które przechodziły
# przez dropna() i trafiały do zbioru oceny.
_target_col = 'target_liquidity_stress'
_target = feat[_target_col].copy()
feat = feat.ffill(limit=5)
feat[_target_col] = _target
assert feat[_target_col].tail(pl.HORIZON).isna().all(), \
    'Ostatnie HORIZON sesji muszą mieć pustą etykietę — brak okna przyszłości.'

# Usuń wiersze gdzie target lub kluczowe cechy są NaN
key_cols = ['btc_log_return', 'wig_log_return', 'target_liquidity_stress']
n_before = len(feat)
feat = feat.dropna(subset=key_cols)
n_after = len(feat)
print(f"\nUsunięto {n_before - n_after} wierszy z NaN w kluczowych kolumnach.")
print(f"Pozostało: {n_after} obserwacji")

# Końcowy NaN po czyszczeniu
remaining_nan = feat.isnull().sum()
remaining_nan = remaining_nan[remaining_nan > 0]
if len(remaining_nan) > 0:
    print("\nPozostałe NaN (kolumny pomocnicze, akceptowalne):")
    print(remaining_nan.to_string())

Analiza brakujących wartości w zbiorze cech:
                         N brakujących  % brakujących
btc_1h_return_std                 1906     100.000000
btc_1h_return_min                 1906     100.000000
btc_1h_return_max                 1906     100.000000
btc_1h_return_mean                1906     100.000000
btc_dist_from_ma200                199      10.440000
cross_corr_90d                      90       4.720000
cross_beta_60d                      60       3.150000
cross_corr_60d                      60       3.150000
cross_relative_strength             60       3.150000
cross_corr_vol_30d                  50       2.620000
wig_amihud_ma30                     30       1.570000
cross_corr_30d                      30       1.570000
btc_realized_vol_30d                30       1.570000
btc_amihud_ma30                     30       1.570000
btc_rolling_kurt_30d                30       1.570000
btc_rolling_skew_30d                30       1.570000
macro_vix_ma30                      2

## 6. Analiza korelacji cech z targetem

In [10]:
# ============================================================
# KROK 7: Korelacja cech z targetem (point-biserial)
# ============================================================

print("Obliczanie korelacji cech z targetem...")

# Wybierz tylko kolumny numeryczne (nie target)
feature_cols = [c for c in feat.columns
                if c not in ['target_liquidity_stress', 'liq_stress_index',
                             'liq_amihud', 'liq_spread', 'liq_turnover_gap',
                             'liq_realized_vol']
                and feat[c].dtype in [np.float64, np.int64, float, int]]

target = feat['target_liquidity_stress'].dropna()

corr_results = []
for col in tqdm(feature_cols, desc="Korelacja z targetem"):
    series = feat[col].dropna()
    # Wyrównaj indeksy
    common_idx = series.index.intersection(target.index)
    if len(common_idx) < 50:
        continue
    x = series.loc[common_idx]
    y = target.loc[common_idx]
    
    # Point-biserial correlation (dla zmiennej binarnej i ciągłej)
    try:
        corr, pval = pointbiserialr(y, x)
        corr_results.append({
            'Cecha': col,
            'Korelacja point-biserial': round(corr, 6),
            '|Korelacja|': round(abs(corr), 6),
            'p-value': round(pval, 6),
            'Istotna (p<0.05)': 'TAK' if pval < 0.05 else 'NIE',
            'N': len(common_idx)
        })
    except Exception:
        pass

corr_df = pd.DataFrame(corr_results).set_index('Cecha')
corr_df = corr_df.sort_values('|Korelacja|', ascending=False)

print("\nTop 20 cech według korelacji z targetem:")
print(corr_df.head(20)[['Korelacja point-biserial', 'p-value', 'Istotna (p<0.05)']].to_string())

corr_df.to_csv(RESULTS_TABLES / '04_feature_target_correlation.csv', float_format='%.6f')
print(f"\nZapisano: {RESULTS_TABLES / '04_feature_target_correlation.csv'}")

Obliczanie korelacji cech z targetem...


Korelacja z targetem:   0%|          | 0/56 [00:00<?, ?it/s]


Top 20 cech według korelacji z targetem:
                        Korelacja point-biserial  p-value Istotna (p<0.05)
Cecha                                                                     
macro_vix                               0.504783 0.000000              TAK
wig_realized_vol_21d                    0.489460 0.000000              TAK
macro_vix_above_30                      0.486334 0.000000              TAK
wig_amihud_ma30                         0.446198 0.000000              TAK
wig_parkinson_vol                       0.415752 0.000000              TAK
macro_vix_ma30                          0.405879 0.000000              TAK
macro_vix_above_40                      0.379375 0.000000              TAK
btc_realized_vol_30d                    0.299470 0.000000              TAK
cross_corr_60d                          0.282588 0.000000              TAK
btc_realized_vol_14d                    0.279843 0.000000              TAK
btc_realized_vol_21d                    0.278629 0.000000 

In [11]:
# ============================================================
# KROK 8: Wykres rankingu cech
# ============================================================

top_n = 25
top_features = corr_df.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))

colors = [
    COLOR_BTC if c.startswith('btc_') else
    COLOR_WIG if c.startswith('wig_') else
    '#27AE60' if c.startswith('cross_') else
    '#8E44AD' if c.startswith('macro_') else '#7F8C8D'
    for c in top_features.index
]

bars = ax.barh(range(len(top_features)), top_features['|Korelacja|'],
               color=colors, alpha=0.8, edgecolor='white', height=0.7)

# Kolorowe wartości: czerwony = pozytywna korelacja z stresem, niebieski = negatywna
for i, (idx, row) in enumerate(top_features.iterrows()):
    corr_val = row['Korelacja point-biserial']
    color_text = '#C0392B' if corr_val > 0 else '#2980B9'
    ax.text(row['|Korelacja|'] + 0.001, i, f"{corr_val:+.4f}",
            va='center', fontsize=8.5, color=color_text)

ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features.index, fontsize=9)
ax.set_xlabel('|Korelacja point-biserial| z targetem')
ax.set_title(f'Top {top_n} cech według siły korelacji z target_liquidity_stress')

# Legenda kolorów grup
legend_patches = [
    mpl.patches.Patch(color=COLOR_BTC, label='BTC features'),
    mpl.patches.Patch(color=COLOR_WIG, label='WIG20 features'),
    mpl.patches.Patch(color='#27AE60', label='Cross-market features'),
    mpl.patches.Patch(color='#8E44AD', label='Macro features'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.invert_yaxis()  # najwyższe na górze

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_12_feature_importance_correlation.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_12_feature_importance_correlation.png")

Zapisano: fig_12_feature_importance_correlation.png


In [12]:
# ============================================================
# KROK 9: Macierz korelacji cech (heatmapa)
# ============================================================

# Wybierz top cechy do heatmapy
top_features_list = corr_df.head(20).index.tolist()
feat_corr_matrix = feat[top_features_list].corr(method='pearson')

fig, ax = plt.subplots(figsize=(14, 12))

mask = np.triu(np.ones_like(feat_corr_matrix), k=1).astype(bool)
sns.heatmap(feat_corr_matrix, ax=ax,
            annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, linewidths=0.3,
            annot_kws={'size': 8},
            square=True)

ax.set_title('Macierz korelacji Pearsona — Top 20 cech', fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_13_feature_correlation_heatmap.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_13_feature_correlation_heatmap.png")

Zapisano: fig_13_feature_correlation_heatmap.png


In [13]:
# ============================================================
# KROK 10: Zapis finalnego zbioru cech
# ============================================================

print("=" * 60)
print("Zapis zbioru cech do pliku")
print("=" * 60)

# Zapisz pełny dataset cech
feat_path = DATA_PROC / 'features_dataset.csv'
feat.to_csv(feat_path, float_format='%.8f')

print(f"Zapisano: {feat_path}")
print(f"  Wymiary: {feat.shape}")
print(f"  Zakres: {feat.index.min().date()} → {feat.index.max().date()}")
print(f"  Kolumny ({len(feat.columns)}):")

# Grupuj cechy
groups = {
    'BTC': [c for c in feat.columns if c.startswith('btc_') and 'target' not in c],
    'WIG20': [c for c in feat.columns if c.startswith('wig_') and 'target' not in c],
    'Cross-market': [c for c in feat.columns if c.startswith('cross_')],
    'Macro': [c for c in feat.columns if c.startswith('macro_')],
    'Target': [c for c in feat.columns if c.startswith('target_')],
}

for group_name, cols in groups.items():
    print(f"  {group_name} ({len(cols)} cech): {cols}")

# Zapisz listę cech do pliku (do użycia w kolejnych notebookach)
feature_list = [c for c in feat.columns if not c.startswith('target_')]
feature_metadata = {
    'all_features': feature_list,
    'btc_features': groups['BTC'],
    'wig_features': groups['WIG20'],
    'cross_features': groups['Cross-market'],
    'macro_features': groups['Macro'],
    'target_cols': groups['Target'],
    'n_features': len(feature_list),
    'n_observations': len(feat),
    'target_balance': {
        'n_positive': int(feat['target_liquidity_stress'].sum()),
        'n_negative': int((feat['target_liquidity_stress'] == 0).sum()),
        'pct_positive': float(feat['target_liquidity_stress'].mean() * 100)
    },
    'date_range': {
        'start': str(feat.index.min().date()),
        'end': str(feat.index.max().date())
    }
}

import json
meta_path = DATA_PROC / 'features_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(feature_metadata, f, indent=2, ensure_ascii=False)

print(f"\nMetadane cech: {meta_path}")
print(f"\nPodsumowanie zbioru cech:")
print(f"  Łącznie cech: {len(feature_list)}")
print(f"  Obserwacji: {len(feat)}")
print(f"  Klasa pozytywna (stress=1): {feature_metadata['target_balance']['pct_positive']:.1f}%")

Zapis zbioru cech do pliku
Zapisano: data/processed/features_dataset.csv
  Wymiary: (1905, 62)
  Zakres: 2019-01-03 → 2026-08-14
  Kolumny (62):
  BTC (27 cech): ['btc_log_return', 'btc_1h_return_mean', 'btc_1h_return_std', 'btc_1h_return_min', 'btc_1h_return_max', 'btc_realized_vol_7d', 'btc_realized_vol_14d', 'btc_realized_vol_21d', 'btc_realized_vol_30d', 'btc_amihud', 'btc_amihud_ma30', 'btc_volume_normalized', 'btc_parkinson_vol', 'btc_rsi_14', 'btc_macd', 'btc_macd_signal', 'btc_macd_hist', 'btc_bb_width', 'btc_rolling_skew_30d', 'btc_rolling_kurt_30d', 'btc_price_above_ma200', 'btc_dist_from_ma200', 'btc_log_return_lag1', 'btc_log_return_lag2', 'btc_log_return_lag3', 'btc_log_return_lag5', 'btc_log_return_lag10']
  WIG20 (11 cech): ['wig_log_return', 'wig_amihud', 'wig_amihud_ma30', 'wig_volume_normalized', 'wig_parkinson_vol', 'wig_realized_vol_21d', 'wig_rsi_14', 'wig_log_return_lag1', 'wig_log_return_lag2', 'wig_log_return_lag3', 'wig_log_return_lag5']
  Cross-market (7 cech)

In [14]:
# ============================================================
# PODSUMOWANIE NOTEBOOKA 04
# ============================================================

print("=" * 60)
print(f"Notebook 04 zakończony: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

saved_files = [
    DATA_PROC / 'features_dataset.csv',
    DATA_PROC / 'features_metadata.json',
    RESULTS_TABLES / '04_feature_target_correlation.csv',
    RESULTS_FIGS / 'fig_12_feature_importance_correlation.png',
    RESULTS_FIGS / 'fig_13_feature_correlation_heatmap.png',
]

print("\nZapisane pliki:")
for f in saved_files:
    exists = f.exists()
    status = 'OK' if exists else 'BRAK'
    size = f"{f.stat().st_size / 1024:.1f} KB" if exists else '---'
    print(f"  [{status}] {f.name:<50} ({size})")

print("\n" + "=" * 60)
print("GOTOWE! Przejdź do Notebook 05 — Modele LSTM i GRU.")
print("=" * 60)

Notebook 04 zakończony: 2026-08-18 20:33:45

Zapisane pliki:
  [OK] features_dataset.csv                               (1183.3 KB)
  [OK] features_metadata.json                             (3.2 KB)
  [OK] 04_feature_target_correlation.csv                  (2.8 KB)
  [OK] fig_12_feature_importance_correlation.png          (373.2 KB)
  [OK] fig_13_feature_correlation_heatmap.png             (1026.0 KB)

GOTOWE! Przejdź do Notebook 05 — Modele LSTM i GRU.
